In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import os

print("Loading data...")
max_features=10000
(X_train,y_train),(X_test,y_test)=imdb.load_data(num_words=max_features)
max_len=500
X_train=sequence.pad_sequences(X_train,maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_len)

print("Building model...")
model=Sequential()
model.add(Embedding(max_features,128,input_length=max_len))
model.add(SimpleRNN(128,activation='tanh'))
model.add(Dense(1,activation="sigmoid"))

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

earlystopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

print("Training model...")
model.fit(X_train,y_train,epochs=3,batch_size=32,validation_split=0.2,callbacks=[earlystopping])
# Just 3 epochs for speed, as early stopping likely doesn't hit soon, and 3 is enough for decent weights without NaNs!

model.save("rnn.keras")
print("Saved my_model.keras successfully.")


Loading data...


/opt/homebrew/lib/python3.11/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


Building model...
Training model...
Epoch 1/3


/opt/homebrew/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - accuracy: 0.6696 - loss: 0.5901 - val_accuracy: 0.7934 - val_loss: 0.4641
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.8073 - loss: 0.4443 - val_accuracy: 0.8002 - val_loss: 0.4500
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.8436 - loss: 0.3829 - val_accuracy: 0.7988 - val_loss: 0.4663
Saved my_model.keras successfully.


In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

model = load_model("rnn.keras")
model.summary()

def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)
    prediction=model.predict(preprocessed_input)
    print(f'PREDICTION IS {prediction}')
    
    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    return sentiment, prediction[0][0]

example_review = "I really like the movie it was great and best movie i have watched. Would recommend to everyone."
sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

# Just print the first couple of weights to ensure they are not nan
weights = model.get_weights()[0]
if np.isnan(weights).any():
    print("ERROR: Weights still contain NaNs!!!")
else:
    print("SUCCESS: Weights are valid!")
# words = "I really like the movie it was great and best movie i have watched. Would recommend to everyone.".lower().split()

# encoded_review = [word_index.get(word, 2) + 3 for word in words]
# print(encoded_review)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,939,077 (15.03 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,626,052 (10.02 MB)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
PREDICTION IS [[0.88053524]]
Review: I really like the movie it was great and best movie i have watched. Would recommend to everyone.
Sentiment: Positive
Prediction Score: 0.8805352449417114
SUCCESS: Weights are valid!
